In [1]:
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, '../data/human_ai_interactions_data')
import haiid

import warnings
warnings.filterwarnings('ignore')

In [2]:
h_bins = 4
b_bins = 10
h_bin_boundaries = []
min_datapoints = 15

In [3]:
# map advice in [-1,1] to probabilities in [0,1] by assigning event Y=1 to label specified by label_1 
# if label_1==None, assigns event Y=1 to a random label per task instance 
def preprocess_data(data, label_1=None):

        #filter data for 'geographic region' and 'perceived accuracy'
        df = data.loc[(data['geographic_region']=='United States') & (data['perceived_accuracy']==80)]
        #select relevant data columns
        df = df[['task_instance_id','participant_id','correct_label', 'advice', 'response_1','response_2']] 

        #assign event Y=1
        if label_1 == None:
            #assigning event Y=1 to a random label per task instance
            #randomness needed since there are 4 labels accross tasks, but each task has two labels -> assign one of them to event Y=1 
            np.random.seed(320)
            task_ids = df['task_instance_id'].unique()
            df_label = pd.DataFrame(task_ids, columns=['task_instance_id'])
            df_label['y'] = np.random.choice(2, len(task_ids)).astype(int) 
            df = df.merge(df_label, how='left', on='task_instance_id' )
        else:
            #assigning event Y=1 to label specified by label_1 
            df['y'] = (df['correct_label']==label_1).astype(int)

        #compute mapping from [-1,1] to [0,1]
        df[['b','h','h+AI']] = (df.loc[:,[ 'advice','response_1', 'response_2']]+1)/ 2.0 
        df.loc[df['y']==0,['b','h','h+AI']] = 1 - df.loc[df['y']==0,['b','h','h+AI']]

        return(df[['task_instance_id','participant_id','h','b','y','h+AI']])

In [4]:
def discretize_human_estimates_equal_width(df):

        #sort calibration data by human estimate h ascending
        df= df.sort_values(by=['h'])
        
        # split calibration data into uniform sized bins
        #find bin boundaries
        lambda_param = 1/h_bins
        bin_bounds = [lambda_param * (i+1) for i in range(h_bins)]

        # set value of h in calibration data to upper bound of each bin
        df.loc[ (df['h']< bin_bounds[0]) | np.isclose(df['h'],bin_bounds[0]), ['h_bin']] = bin_bounds[0]
        for i in range(1, len(bin_bounds)):
            df.loc[((df['h']< bin_bounds[i]) | np.isclose(df['h'],bin_bounds[i])) & (df['h']> bin_bounds[i-1]),['h_bin']] = bin_bounds[i]

        #set data and h bin boundaries
        return(df)
        # h_bin_boundaries = bin_bounds

In [5]:
# load data
def get_task_data(task_name):
        df = haiid.load_dataset("../data/human_ai_interactions_data")
        # get specified task data
        task_df = haiid.load_task(df, task_name)
        if task_name =='census':
            task_data = preprocess_data(task_df, '>=50k')
        elif task_name =='sarcasm':
            task_data = preprocess_data(task_df, 'sarcasm')
        else:
            task_data  = preprocess_data(task_df)
        
        task_data = discretize_human_estimates_equal_width(task_data)

        return (task_data)

In [6]:
#compute probabilities and density mass of each bin (h,b)
def compute_cell_prob_matrix(df):
        lambda_param = 1/b_bins
        
        # partition model risk estimate space into B bins,
        # set new column in df with the max bin value (indicates in which bin each data point is))
        df['b_bin'] = (df['b']// lambda_param) +1
        df.loc[df['b_bin']==b_bins+1, ['b_bin']] = b_bins
        df['b_bin'] *= lambda_param
        df['b_bin'] = df['b_bin'].round(3) 

        #drop original model risk estimates
        # df = df.drop(columns=['b','h+AI'])
        #compute probability Y=1 and density mass in each bin
        df_cell_prob = df.groupby(by=['b_bin','h_bin'])['y'].mean().unstack()
        df_cell_mass = df.groupby(by=['b_bin','h_bin'])['y'].count().unstack()

        return(df, df_cell_prob, df_cell_mass)
        #update task data with b_bin value for each data point
        # self.task_data["b_bin"] = df['b_bin']

In [7]:
#computes expected and average alignment error
def check_alignment(df_cell_prob, df_cell_mass):
        #compute max and average alignment violations
        max_aligment = 0.0
        sum_aligment = 0.0
        num_summants = 0.0
        disaligned_cells = set({})

        for h in range(df_cell_prob.columns.shape[0]):
            for b in range(df_cell_prob.index.shape[0]):
                for h1 in range(h+1):
                    for b1 in range(b+1):
                        num_summants +=1
                        # check misalignment of the pair of cells if enough datapoints in each cells
                        if (df_cell_mass.iat[b,h]>=min_datapoints) & (df_cell_mass.iat[b1,h1]>=min_datapoints):
                            alignment = max(0.0, df_cell_prob.iat[b1,h1] - df_cell_prob.iat[b,h] )
                            max_aligment = max(max_aligment, alignment) 
                            if alignment > 0.0:
                                sum_aligment += alignment
                                                                
        if num_summants > 0: 
            avg_alignment = sum_aligment / num_summants
        else:
            avg_alignment = 0.0 
        
        mae = max_aligment 
        eae = avg_alignment

        return(eae, mae)

In [8]:
alignment_error = {}

#preprocess data, compute cell probabilities and density mass, and compute alignment errors
#art task does not have a label that can be used to assign event Y=1, so we assign event Y=1 to a random label for each task instance
df_art = get_task_data('art')
df_art, df_cell_prob, df_cell_mass = compute_cell_prob_matrix(df_art)
eae, mae = check_alignment(df_cell_prob, df_cell_mass)
alignment_error['art'] = {"EAE": eae, "MAE": mae}

#census task: assign event Y=1 to label '>=50k'
df_census = get_task_data('census')
df_census, df_cell_prob, df_cell_mass = compute_cell_prob_matrix(df_census)
eae, mae = check_alignment(df_cell_prob, df_cell_mass)
alignment_error['census'] = {"EAE": eae, "MAE": mae}

#sarcasm task: assign event Y=1 to label 'sarcasm'
df_sarcasm = get_task_data('sarcasm')
df_sarcasm, df_cell_prob, df_cell_mass = compute_cell_prob_matrix(df_sarcasm)
eae, mae = check_alignment(df_cell_prob, df_cell_mass)
alignment_error['sarcasm'] = {"EAE": eae, "MAE": mae}

#cities task: does not have a label that can be used to assign event Y=1, so we assign event Y=1 to a random label for each task instance
df_cities = get_task_data('cities')
df_cities, df_cell_prob, df_cell_mass = compute_cell_prob_matrix(df_cities)
eae, mae = check_alignment(df_cell_prob, df_cell_mass)
alignment_error['cities'] = {"EAE": eae, "MAE": mae}    
 

In [9]:
df_alignment = pd.DataFrame(alignment_error).T
df_alignment

,EAE,MAE
art,0.000922,0.108232
census,0.029840,0.673132
sarcasm,0.000968,0.129716
cities,0.000272,0.065361


### Save Preprocessed Data

In [10]:
df_art.to_csv('../data/human_ai_interactions_data/processed/data_art.csv', index=False)
df_census.to_csv('../data/human_ai_interactions_data/processed/data_census.csv', index=False)
df_sarcasm.to_csv('../data/human_ai_interactions_data/processed/data_sarcasm.csv', index=False)
df_cities.to_csv('../data/human_ai_interactions_data/processed/data_cities.csv', index=False)